# 03 — Robustness Checks

Three checks to ensure H1 findings are not artifacts:

1. **Permutation test** — are metric values more extreme than random label shuffling?
2. **Downsampling test** — do findings hold when all outlets are equalized to smallest size?
3. **Sensitivity analysis** — do findings change with different merge thresholds?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("No .git found")

PROJECT_ROOT = find_project_root(Path.cwd())
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "agenda_distortion"
OUTPUT_DIR = EXPERIMENT_DIR / "outputs"

for p in [str(EXPERIMENT_DIR), str(PROJECT_ROOT / "1a_BERTopic")]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [ ]:
from modeling import load_iteration

ITERATION_ID = "v1"
result = load_iteration(OUTPUT_DIR, ITERATION_ID)
merged_articles = result.merged_articles

print(f"Loaded: {len(merged_articles):,} articles, {result.n_topics} topics")

## 1. Permutation Test

Shuffle outlet labels → recompute JSD → compare to observed.
If p < 0.05, the observed divergence is unlikely to be random.

In [ ]:
from robustness import permutation_test

print("Running permutation test (n=500)... this takes a while.")
perm_result = permutation_test(
    merged_articles,
    metric="jsd_vs_tagesschau",
    n_perms=500,
)

print("\nPermutation Test Results (JSD)")
print("=" * 60)
display(perm_result.summary())

perm_result.summary().to_csv(OUTPUT_DIR / ITERATION_ID / "robustness_permutation.csv", index=False)

## 2. Downsampling Test

Equalize all outlets to the size of the smallest (Antispiegel ~565).
If metrics remain directionally stable, findings are not driven by size.

In [ ]:
from robustness import downsample_test

print("Running downsampling test (n=50 resamples)...")
downsample_df = downsample_test(merged_articles, n_resamples=50)

print("\nDownsampled Metrics (mean ± std, equalized corpus sizes)")
print("=" * 80)

# Show key columns
display_cols = [
    "outlet_label", "n_per_resample",
    "jsd_vs_tagesschau_mean", "jsd_vs_tagesschau_std",
    "spearman_rho_mean", "spearman_rho_std",
    "entropy_normalized_mean", "entropy_normalized_std",
]
display(downsample_df[display_cols])

downsample_df.to_csv(OUTPUT_DIR / ITERATION_ID / "robustness_downsample.csv", index=False)

## 3. Sensitivity Analysis — Merge Threshold

Re-merge models at different `min_similarity` values (0.5, 0.6, 0.7, 0.8).
If outlet rankings remain stable, the merge threshold is not driving results.

**Warning**: This re-runs the full merge + transform pipeline 4 times. Expect ~10min.

In [ ]:
from robustness import sensitivity_analysis

print("Running sensitivity analysis across merge thresholds...")
sensitivity_df = sensitivity_analysis(
    PROJECT_ROOT,
    similarity_values=[0.5, 0.6, 0.7, 0.8],
)

# Show JSD across thresholds
print("\nJSD vs Tagesschau across merge thresholds")
print("=" * 60)
pivot = sensitivity_df.pivot_table(
    index="outlet_label",
    columns="min_similarity",
    values="jsd_vs_tagesschau",
)
display(pivot)

# Topic counts
print("\nMerged topic count by threshold:")
for sim in sorted(sensitivity_df["min_similarity"].unique()):
    n = sensitivity_df.loc[sensitivity_df["min_similarity"] == sim, "n_merged_topics"].iloc[0]
    print(f"  min_similarity={sim}: {n} topics")

sensitivity_df.to_csv(OUTPUT_DIR / ITERATION_ID / "robustness_sensitivity.csv", index=False)

## Summary

Fill in after running:

| Check | Result | Pass? |
|-------|--------|-------|
| Permutation p < 0.05 | [outlets] | |
| Downsampled metrics stable | [Y/N] | |
| Sensitivity to merge threshold | [stable/fragile] | |

**If all checks pass** → document in `findings.md` and `iteration_log.md`.

**If checks fail** → iterate in notebook 01 with different parameters.